# Lab 4 — Build the Audience, Then Buy It
Name: Anna Zaporozhets | Group: KM-42

## Setup

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

DATA = kagglehub.dataset_download(
    "shkarupylomaxim/adtech-lab4-tv-audiences/versions/2"
)

programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")

survey = pd.read_csv(
    f"{DATA}/purchase_survey.txt",
    sep="\t",
    dtype=str,
    keep_default_na=False
)

survey_layout = pd.read_csv(
    f"{DATA}/purchase_survey_layout.txt",
    sep="\t",
    dtype=str
)

acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

print("All twelve datasets loaded successfully.")

Using Colab cache for faster access to the 'adtech-lab4-tv-audiences' dataset.
All twelve datasets loaded successfully.


## Task 1.0 — Clean the survey file

In [ ]:
# Prepare the survey data.
original = survey[["RESPID", "KEYWORD", "VALUE"]].fillna("").astype(str)

survey_data = original.apply(lambda col: col.str.strip())

# Normalize respondent IDs.
def normalize_ids(ids):
    return (
        ids.fillna("").astype(str).str.strip()
        .str.replace(r"\.0+$", "", regex=True)
        .str.lstrip("0")
    )

survey_data["RESPID"] = normalize_ids(survey_data["RESPID"])
survey_data["KEYWORD"] = survey_data["KEYWORD"].str.upper()
survey_data["VALUE"] = survey_data["VALUE"].str.upper()

yes_values = {"1", "1.0", "Y", "YES", "TRUE"}
no_values = {"0", "0.0", "N", "NO"}

layout = survey_layout.loc[
    survey_layout["RECTYPE"] == "B",
    ["KEYWORD", "SAMPLE"]
].copy()

layout["KEYWORD"] = layout["KEYWORD"].str.strip().str.upper()
valid_keywords = set(layout["KEYWORD"])

# Identify R and L panel members aged 18+.
panel_ids = set(
    normalize_ids(
        viewer_attributes.loc[
            viewer_attributes["attr_id"].isin([101, 102]),
            "respondent_id"
        ]
    )
)

ages = viewer_attributes.loc[
    viewer_attributes["attr_id"] == 300,
    ["respondent_id", "value"]
].copy()

ages["RESPID"] = normalize_ids(ages["respondent_id"])
ages["age"] = pd.to_numeric(ages["value"], errors="coerce")

adult_ids = set(
    ages.loc[ages["age"] >= 18, "RESPID"]
) & panel_ids

# Keep valid yes answers from adults.
valid_yes = (
    survey_data["RESPID"].isin(adult_ids)
    & survey_data["KEYWORD"].isin(valid_keywords)
    & survey_data["VALUE"].isin(yes_values)
)

adult_yes = survey_data.loc[valid_yes].copy()
adult_yes["VALUE"] = "1"

duplicate_rows = adult_yes.duplicated(
    subset=["RESPID", "KEYWORD"]
).sum()

adult_yes = adult_yes.drop_duplicates(
    subset=["RESPID", "KEYWORD"]
)

# Resolve conflicting pair counts.
pair_codes = {
    "X710101": 1,
    "X710102": 2,
    "X710103": 3
}

pair_rows = adult_yes.loc[
    adult_yes["KEYWORD"].isin(pair_codes)
].copy()

pair_rows["pair_count"] = pair_rows["KEYWORD"].map(pair_codes)

pair_counts = pair_rows.groupby("RESPID")["KEYWORD"].nunique()
conflict_ids = set(pair_counts[pair_counts > 1].index)

two_pair_counts = len(conflict_ids)
conflict_rows = pair_rows["RESPID"].isin(conflict_ids).sum()

highest_pairs = (
    pair_rows.sort_values("pair_count")
    .drop_duplicates("RESPID", keep="last")
)

clean_yes = pd.concat([
    adult_yes.loc[
        ~adult_yes["KEYWORD"].isin(pair_codes)
    ],
    highest_pairs.drop(columns="pair_count")
], ignore_index=True)

# A pair count or brand answer also identifies a buyer.
buyer_ids = set(clean_yes["RESPID"])

# Restore implied Any answers.
any_ids = set(
    clean_yes.loc[
        clean_yes["KEYWORD"] == "X710100", "RESPID"
    ]
)

missing_any_ids = buyer_ids - any_ids

implied_any = pd.DataFrame({
    "RESPID": sorted(missing_any_ids),
    "KEYWORD": "X710100",
    "VALUE": "1"
})

clean_yes = pd.concat(
    [clean_yes, implied_any],
    ignore_index=True
)

# Create one row per adult panel member.
purchase_panel = pd.DataFrame({
    "RESPID": sorted(adult_ids)
})

purchase_panel["buyer"] = np.where(
    purchase_panel["RESPID"].isin(buyer_ids),
    "yes", "no"
)

pair_by_person = highest_pairs.set_index("RESPID")["pair_count"]

purchase_panel["pairs"] = (
    purchase_panel["RESPID"]
    .map(pair_by_person)
    .map({1: "1", 2: "2", 3: "3+"})
)

purchase_panel.loc[
    purchase_panel["buyer"] == "no", "pairs"
] = "0"

# Identify the three brands.
brand_codes = {
    "stride": "X710201",
    "kestrel": "X710202",
    "easywalk": "X710203"
}

for brand, keyword in brand_codes.items():
    brand_ids = set(
        clean_yes.loc[
            clean_yes["KEYWORD"] == keyword, "RESPID"
        ]
    )

    purchase_panel[brand] = np.where(
        purchase_panel["RESPID"].isin(brand_ids),
        "yes", "no"
    )

# Required results.
print("Survey data rows:", len(survey))
print("Clean yes answers:", len(clean_yes))
print("Adult panel members:", len(purchase_panel))
print("Non-buyers:", (purchase_panel["buyer"] == "no").sum())
print("People with two pair counts:", two_pair_counts)

# Table 1: Data issues and corrections.
whitespace_rows = original.ne(
    original.apply(lambda col: col.str.strip())
).any(axis=1).sum()

id_format_rows = (
    original["RESPID"].str.strip() != survey_data["RESPID"]
).sum()

lowercase_rows = (
    original["KEYWORD"].str.strip()
    != survey_data["KEYWORD"]
).sum()

alternative_yes = survey_data["VALUE"].isin(
    yes_values - {"1"}
).sum()

negative_answers = survey_data["VALUE"].isin(no_values).sum()

invalid_answers = (
    ~survey_data["VALUE"].isin(yes_values | no_values)
).sum()

unknown_keywords = (
    ~survey_data["KEYWORD"].isin(valid_keywords)
).sum()

valid_codes_and_yes = (
    survey_data["VALUE"].isin(yes_values)
    & survey_data["KEYWORD"].isin(valid_keywords)
)

invalid_respondents = (
    valid_codes_and_yes
    & ~survey_data["RESPID"].isin(adult_ids)
).sum()

issues = pd.DataFrame([
    ["Extra whitespace", whitespace_rows, "Trimmed"],
    ["Incorrect ID format", id_format_rows, "Normalized IDs"],
    ["Lowercase keywords", lowercase_rows, "Converted to uppercase"],
    ["Alternative yes values", alternative_yes, "Accepted as yes"],
    ["Negative answers", negative_answers, "Removed"],
    ["Invalid or empty values", invalid_answers, "Removed"],
    ["Unknown keywords", unknown_keywords, "Removed"],
    ["Invalid respondents", invalid_respondents, "Removed"],
    ["Duplicate answers", duplicate_rows, "Kept one answer"],
    ["Conflicting pair counts", conflict_rows, "Kept the higher count"],
    ["Missing Any answers", len(missing_any_ids), "Inferred from purchases"]
], columns=["Issue", "Rows", "Action"])

display(issues)

# Table 2: Compare cleaned counts with SAMPLE.
layout["SAMPLE"] = pd.to_numeric(
    layout["SAMPLE"], errors="coerce"
)

clean_counts = clean_yes.groupby("KEYWORD")["RESPID"].nunique()

layout["Your count"] = (
    layout["KEYWORD"]
    .map(clean_counts)
    .fillna(0)
    .astype(int)
)

display(layout[["KEYWORD", "Your count", "SAMPLE"]])

print(
    "All SAMPLE counts match:",
    (layout["Your count"] == layout["SAMPLE"]).all()
)

# Compare the raw and cleaned buyer counts.
raw_kestrel = (
    (survey["KEYWORD"] == "X710202")
    & (survey["VALUE"] == "1")
).sum()

raw_buyers = (
    (survey["KEYWORD"] == "X710100")
    & (survey["VALUE"] == "1")
).sum()

clean_kestrel = (purchase_panel["kestrel"] == "yes").sum()
clean_buyers = (purchase_panel["buyer"] == "yes").sum()

raw_share = raw_buyers / len(adult_ids) * 100
clean_share = clean_buyers / len(adult_ids) * 100

print("Raw Kestrel count:", raw_kestrel)
print("Clean Kestrel count:", clean_kestrel)
print("Kestrel count difference:", raw_kestrel - clean_kestrel)

print("Raw buyer share (%):", round(raw_share, 1))
print("Clean buyer share (%):", round(clean_share, 1))
print("Buyer share difference (pp):", round(raw_share - clean_share, 1))

# Check whether any buyers still have an unknown pair count.
unknown_pairs = (
    (purchase_panel["buyer"] == "yes")
    & purchase_panel["pairs"].isna()
).sum()

print("Buyers without reported pair count:", unknown_pairs)

Survey data rows: 18349
Clean yes answers: 15635
Adult panel members: 8010
Non-buyers: 3231
People with two pair counts: 113


,Issue,Rows,Action
0,Extra whitespace,932,Trimmed
1,Incorrect ID format,882,Normalized IDs
2,Lowercase keywords,734,Converted to uppercase
3,Alternative yes values,1194,Accepted as yes
4,Negative answers,619,Removed
5,Invalid or empty values,126,Removed
6,Unknown keywords,741,Removed
7,Invalid respondents,423,Removed
8,Duplicate answers,933,Kept one answer
9,Conflicting pair counts,226,Kept the higher count


,KEYWORD,Your count,SAMPLE
2,X710100,4779,4779
3,X710101,1810,1810
4,X710102,1308,1308
5,X710103,1661,1661
7,X710201,1563,1563
8,X710202,354,354
9,X710203,1050,1050
10,X710298,2739,2739
11,X710299,371,371


All SAMPLE counts match: True
Raw Kestrel count: 383
Clean Kestrel count: 354
Kestrel count difference: 29
Raw buyer share (%): 52.5
Clean buyer share (%): 59.7
Buyer share difference (pp): -7.2
Buyers without reported pair count: 0


**Answer:**

The original survey had several problems. Some respondent IDs were incorrectly formatted, the answer values were inconsistent, and there were also duplicates and invalid records. I cleaned the IDs and answers, removed invalid rows and kept only adult panel members. For the 113 people who reported two different pair counts, I kept the higher one.

I also found 238 people who had reported a pair count or a brand but had no "Any" answer. I counted them as buyers because their other answers already confirmed a purchase.

After cleaning, I got 4,779 buyers out of 8,010 adults, including 354 Kestrel buyers. All the cleaned counts matched the SAMPLE values from the layout.

Without cleaning, counting the raw entries would give 383 Kestrel records instead of 354 buyers, which is 29 too many. The share of shoe buyers would also be underestimated: 52.5% instead of 59.7%, a difference of 7.2 percentage points.

## Task 1.1 — Four definitions

In [ ]:
# Select October daily weights for R and L panel members.
october_days = pd.date_range("2024-10-01", "2024-10-31")

weights = daily_weights.loc[
    daily_weights["resp_type"].isin(["R", "L"]),
    ["respondent_id", "broadcast_date", "weight"]
].copy()

weights["broadcast_date"] = pd.to_datetime(
    weights["broadcast_date"]
).dt.normalize()

weights = weights.loc[
    weights["broadcast_date"].isin(october_days)
].copy()

weights["RESPID"] = normalize_ids(weights["respondent_id"])

# Add age and purchase information for adult panel members.
adult_profiles = purchase_panel.merge(
    ages[["RESPID", "age"]].drop_duplicates("RESPID"),
    on="RESPID",
    how="left",
    validate="one_to_one"
)

adult_weights = weights.merge(
    adult_profiles,
    on="RESPID",
    how="inner",
    validate="many_to_one"
)

# Average the daily audience sizes over 31 days.
def audience_size(mask):
    daily_size = (
        adult_weights.loc[mask]
        .groupby("broadcast_date")["weight"]
        .sum()
        .reindex(october_days, fill_value=0)
    )
    return daily_size.mean() / 1_000_000

adults_18 = audience_size(adult_weights["age"] >= 18)
shoe_buyers = audience_size(adult_weights["buyer"] == "yes")
heavy_buyers = audience_size(adult_weights["pairs"] == "3+")
kestrel_buyers = audience_size(adult_weights["kestrel"] == "yes")
adults_25_54 = audience_size(adult_weights["age"].between(25, 54))

kestrel_panel = (purchase_panel["kestrel"] == "yes").sum()

print("Adults 18+ (M):", f"{adults_18:.1f}")
print("Shoe buyers (M):", f"{shoe_buyers:.1f}")
print("Heavy buyers, 3+ pairs (M):", f"{heavy_buyers:.1f}")
print("Kestrel buyers (M):", f"{kestrel_buyers:.1f}")
print("Adults 25-54 (M):", f"{adults_25_54:.1f}")
print("Kestrel buyers in panel (n):", kestrel_panel)

Adults 18+ (M): 252.6
Shoe buyers (M): 148.1
Heavy buyers, 3+ pairs (M): 49.6
Kestrel buyers (M): 10.0
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 354


## Task 1.2 — Who are they?

In [ ]:
# Add sex information to the adult audience.
sex_info = viewer_attributes.loc[
    viewer_attributes["attr_id"].isin([201, 202]),
    ["respondent_id", "attr_id"]
].copy()

sex_info["RESPID"] = normalize_ids(sex_info["respondent_id"])
sex_info["sex"] = sex_info["attr_id"].map({
    201: "Men",
    202: "Women"
})

sex_info = sex_info[["RESPID", "sex"]].drop_duplicates("RESPID")

profile_weights = adult_weights.merge(
    sex_info,
    on="RESPID",
    how="left",
    validate="many_to_one"
)

# Create age groups.
profile_weights["age_group"] = pd.cut(
    profile_weights["age"],
    bins=[17, 34, 54, float("inf")],
    labels=["18-34", "35-54", "55+"]
)

# Define the audiences.
audiences = {
    "Adults 18+": profile_weights["age"] >= 18,
    "Shoe buyers": profile_weights["buyer"] == "yes",
    "Heavy buyers": profile_weights["pairs"] == "3+",
    "Kestrel buyers": profile_weights["kestrel"] == "yes",
    "Adults 25-54": profile_weights["age"].between(25, 54)
}

# Define age and sex groups.
groups = {
    "18-34": profile_weights["age_group"] == "18-34",
    "35-54": profile_weights["age_group"] == "35-54",
    "55+": profile_weights["age_group"] == "55+",
    "Men": profile_weights["sex"] == "Men",
    "Women": profile_weights["sex"] == "Women"
}

# Calculate average daily weighted size.
def average_size(data):
    return (
        data.groupby("broadcast_date")["weight"]
        .sum()
        .reindex(october_days, fill_value=0)
        .mean()
    )

# Calculate group shares for all adults.
adult_total = average_size(profile_weights)

adult_shares = {
    group: average_size(profile_weights.loc[mask]) / adult_total
    for group, mask in groups.items()
}

# Calculate shares and indexes for each audience.
results = []

for audience, audience_mask in audiences.items():
    audience_data = profile_weights.loc[audience_mask]
    audience_total = average_size(audience_data)

    for group, group_mask in groups.items():
        group_data = profile_weights.loc[
            audience_mask & group_mask
        ]

        share = average_size(group_data) / audience_total
        index = share / adult_shares[group] * 100

        results.append({
            "Audience": audience,
            "Group": group,
            "Share": round(share, 3),
            "Index": round(index, 1)
        })

profile_table = pd.DataFrame(results)

display(profile_table)

,Audience,Group,Share,Index
0,Adults 18+,18-34,0.288,100.0
1,Adults 18+,35-54,0.319,100.0
2,Adults 18+,55+,0.393,100.0
3,Adults 18+,Men,0.488,100.0
4,Adults 18+,Women,0.512,100.0
5,Shoe buyers,18-34,0.281,97.5
6,Shoe buyers,35-54,0.336,105.3
7,Shoe buyers,55+,0.383,97.5
8,Shoe buyers,Men,0.485,99.3
9,Shoe buyers,Women,0.515,100.7


**Answer:**

I would choose heavy athletic-shoe buyers (3+ pairs) for Kestrel because this is the audience the brand wants to reach. It is also much larger than the group of existing Kestrel buyers (49.6M vs. 10.0M), so there are more potential customers.

Athletic-shoe buyers look similar to all adults because buying shoes is quite common across different age groups and both sexes. Most of their indexes are close to 100, so this definition does not identify a very specific audience.

One limitation of fused purchase data is that it does not directly show whether people watching a particular program actually bought the product. Their purchase attributes are estimated using demographic information.

## Task 2.1 — Programs by audience

In [ ]:
# Select the eight programs.
selected_programs = [
    ("NBC", "NFL Football", "Sunday Night Football (NBC)"),
    ("ABC", "NFL Football", "Monday Night Football (ABC)"),
    ("ABC", "Dancing with the Stars", "Dancing with the Stars (ABC)"),
    ("CBS", "NCIS", "NCIS (CBS)"),
    ("FNC", "Hannity", "Hannity (FNC)"),
    ("HGTV", "Unsellable Houses", "Unsellable Houses (HGTV)"),
    ("FOOD", "Halloween Wars", "Halloween Wars (FOOD)"),
    ("FX", "Grotesquerie", "Grotesquerie (FX)")
]

program_names = {
    (network, name): label
    for network, name, label in selected_programs
}

# Convert timestamps to Eastern time.
def eastern_time(values):
    result = pd.to_datetime(values)
    if result.dt.tz is not None:
        result = result.dt.tz_convert("America/New_York")
        result = result.dt.tz_localize(None)
    return result

# Select October prime-time telecasts.
prime = programs.copy()

prime["broadcast_date"] = pd.to_datetime(
    prime["broadcast_date"]
).dt.normalize()

for col in ["start_ts", "end_ts"]:
    prime[col] = eastern_time(prime[col])

prime = prime.loc[
    prime["broadcast_date"].isin(october_days)
    & prime["daypart"].eq("Prime")
].copy()

prime["program_key"] = list(zip(
    prime["network_code"], prime["program_name"]
))

selected_prime = prime.loc[
    prime["program_key"].isin(program_names)
].copy()

assert selected_prime["program_key"].nunique() == 8, (
    "Some programs are missing from the schedule."
)

# Prepare live viewing records.
live_viewings = viewings.loc[
    viewings["resp_type"].isin(["R", "L"])
    & viewings["playback"].eq("Live")
    & viewings["network_code"].isin(
        [item[0] for item in selected_programs]
    ),
    ["respondent_id", "network_code", "start_ts", "end_ts"]
].copy()

live_viewings["RESPID"] = normalize_ids(
    live_viewings["respondent_id"]
)

for col in ["start_ts", "end_ts"]:
    live_viewings[col] = eastern_time(live_viewings[col])

live_by_network = {
    network: data
    for network, data in live_viewings.groupby("network_code")
}

# Prepare daily audience weights.
audience_weights = adult_weights[
    ["RESPID", "broadcast_date", "weight", "pairs", "kestrel"]
].copy()

day_weights = {
    day: data.set_index("RESPID")
    for day, data in audience_weights.groupby("broadcast_date")
}

# Calculate the average-minute ratings for each telecast.
rating_rows = []

for telecast in selected_prime.itertuples(index=False):
    day = telecast.broadcast_date
    start = telecast.start_ts
    end = telecast.end_ts
    duration = (end - start).total_seconds() / 60

    network_data = live_by_network[telecast.network_code]

    watched = network_data.loc[
        (network_data["start_ts"] < end)
        & (network_data["end_ts"] > start)
    ].copy()

    watched["minutes"] = (
        watched["end_ts"].clip(upper=end)
        - watched["start_ts"].clip(lower=start)
    ).dt.total_seconds() / 60

    minutes_per_person = (
        watched.groupby("RESPID")["minutes"]
        .sum()
        .clip(upper=duration)
    )

    panel_day = day_weights[day]

    viewers = panel_day.join(
        minutes_per_person.rename("minutes"),
        how="inner"
    )

    # Average-minute viewers for each audience.
    adult_viewers = (
        viewers["weight"] * viewers["minutes"]
    ).sum() / duration

    heavy_data = viewers.loc[viewers["pairs"] == "3+"]

    heavy_viewers = (
        heavy_data["weight"] * heavy_data["minutes"]
    ).sum() / duration

    kestrel_data = viewers.loc[viewers["kestrel"] == "yes"]

    kestrel_viewers = (
        kestrel_data["weight"] * kestrel_data["minutes"]
    ).sum() / duration

    # Daily audience sizes.
    adult_universe = panel_day["weight"].sum()

    heavy_universe = panel_day.loc[
        panel_day["pairs"] == "3+", "weight"
    ].sum()

    kestrel_universe = panel_day.loc[
        panel_day["kestrel"] == "yes", "weight"
    ].sum()

    rating_rows.append({
        "telecast_id": telecast.telecast_id,
        "adult_viewers": adult_viewers,
        "heavy_viewers": heavy_viewers,
        "adult_rating": adult_viewers / adult_universe * 100,
        "heavy_rating": heavy_viewers / heavy_universe * 100,
        "kestrel_rating": kestrel_viewers / kestrel_universe * 100
    })

telecast_ratings = pd.DataFrame(rating_rows)

selected_prime = selected_prime.merge(
    telecast_ratings,
    on="telecast_id",
    validate="one_to_one"
)

# Prepare the rate card.
rates = rate_card.copy()
rates["program_name"] = rates["program_name"].fillna("").str.strip()

program_rates = rates.loc[
    rates["program_name"] != ""
]

time_rates = rates.loc[
    rates["program_name"] == ""
]

def clock_minutes(value):
    hour, minute = map(int, value.split(":"))
    return hour * 60 + minute - 360

def matches_weekday(days, weekday):
    if isinstance(days, (list, tuple, np.ndarray)):
        return weekday in [int(x) for x in days]
    if isinstance(days, str):
        values = days.strip("[]").replace(" ", "").split(",")
        return str(weekday) in values
    return False

# Price each telecast's 30-second spot.
def get_price(telecast):
    network = telecast.network_code
    name = telecast.program_name

    program_match = program_rates.loc[
        (program_rates["network_code"] == network)
        & (program_rates["program_name"] == name)
    ]

    if not program_match.empty:
        return program_match.iloc[0]["cost_30s_usd"]

    day = telecast.broadcast_date
    minute = (
        telecast.start_ts - (day + pd.Timedelta(hours=6))
    ).total_seconds() / 60

    network_rates = time_rates.loc[
        time_rates["network_code"] == network
    ]

    matching_rates = network_rates.loc[
        network_rates.apply(
            lambda row:
                matches_weekday(row["weekdays"], day.weekday())
                and clock_minutes(row["start_time"]) <= minute
                < clock_minutes(row["end_time"]),
            axis=1
        )
    ]

    assert len(matching_rates) == 1, (
        f"Rate not found for {name} ({network}) on {day.date()}"
    )

    return matching_rates.iloc[0]["cost_30s_usd"]

selected_prime["cost_30"] = [
    get_price(t) for t in selected_prime.itertuples(index=False)
]

# Average the results across each program's telecasts.
program_summary = (
    selected_prime.groupby(
        ["network_code", "program_name"], as_index=False
    )
    .agg(
        adult_rating=("adult_rating", "mean"),
        heavy_rating=("heavy_rating", "mean"),
        kestrel_rating=("kestrel_rating", "mean"),
        adult_viewers=("adult_viewers", "mean"),
        heavy_viewers=("heavy_viewers", "mean"),
        cost_30=("cost_30", "mean")
    )
)

program_summary["Program"] = [
    program_names[(network, name)]
    for network, name in zip(
        program_summary["network_code"],
        program_summary["program_name"]
    )
]

program_summary["Composition"] = (
    program_summary["heavy_viewers"]
    / program_summary["adult_viewers"]
)

program_summary["Index"] = (
    program_summary["heavy_rating"]
    / program_summary["adult_rating"] * 100
)

program_summary["Adults CPM ($)"] = (
    program_summary["cost_30"]
    / program_summary["adult_viewers"] * 1000
)

program_summary["Target CPM ($)"] = (
    program_summary["cost_30"]
    / program_summary["heavy_viewers"] * 1000
)

program_summary["Target viewers (000s)"] = (
    program_summary["heavy_viewers"] / 1000
)

# Display the required table.
program_table = program_summary.rename(columns={
    "adult_rating": "Adults rating (%)",
    "heavy_rating": "Target rating (%)",
    "cost_30": "Cost per :30 ($)"
})[[
    "Program",
    "Adults rating (%)",
    "Target rating (%)",
    "Target viewers (000s)",
    "Composition",
    "Index",
    "Cost per :30 ($)",
    "Adults CPM ($)",
    "Target CPM ($)"
]]

program_order = [item[2] for item in selected_programs]

program_table = (
    program_table.set_index("Program")
    .loc[program_order]
    .reset_index()
)

display(program_table.style.format({
    "Adults rating (%)": "{:.2f}",
    "Target rating (%)": "{:.2f}",
    "Target viewers (000s)": "{:,.1f}",
    "Composition": "{:.3f}",
    "Index": "{:.1f}",
    "Cost per :30 ($)": "{:,.0f}",
    "Adults CPM ($)": "{:,.0f}",
    "Target CPM ($)": "{:,.0f}"
}))

print("Target universe (M):", f"{heavy_buyers:.1f}")

,Program,Adults rating (%),Target rating (%),Target viewers (000s),Composition,Index,Cost per :30 ($),Adults CPM ($),Target CPM ($)
0,Sunday Night Football (NBC),5.82,5.79,"2,865.0",0.195,99.3,"487,200",33,170
1,Monday Night Football (ABC),8.93,8.36,"4,149.1",0.184,93.6,"723,600",32,174
2,Dancing with the Stars (ABC),3.25,3.14,"1,564.2",0.191,96.8,"158,600",19,101
3,NCIS (CBS),2.21,2.43,"1,206.3",0.216,110.1,"178,800",32,148
4,Hannity (FNC),1.09,0.84,416.9,0.152,77.1,"39,900",15,96
5,Unsellable Houses (HGTV),0.50,0.64,317.2,0.249,126.8,"12,900",10,41
6,Halloween Wars (FOOD),0.28,0.33,161.9,0.228,116.2,"9,700",14,60
7,Grotesquerie (FX),0.26,0.25,124.2,0.188,95.9,"7,592",11,61


Target universe (M): 49.6


## Task 2.2 — Index, volume or price?

In [ ]:
highest_index = program_summary.loc[
    program_summary["Index"].idxmax(), "Program"
]

most_viewers = program_summary.loc[
    program_summary["heavy_viewers"].idxmax(), "Program"
]

lowest_cpm = program_summary.loc[
    program_summary["Target CPM ($)"].idxmin(), "Program"
]

print("Highest index (program):", highest_index)
print("Most target viewers (program):", most_viewers)
print("Lowest target CPM (program):", lowest_cpm)

Highest index (program): Unsellable Houses (HGTV)
Most target viewers (program): Monday Night Football (ABC)
Lowest target CPM (program): Unsellable Houses (HGTV)


**Answer:**

The results are not the same for all three measures. Unsellable Houses (HGTV) has the highest index (126.8) and the lowest target CPM (USD 41), while Monday Night Football (ABC) delivers the most heavy buyers, about 4.15 million.

I would explain to the client that a high index does not always mean the best advertising choice. It shows that the target audience is well represented, but it does not tell us how many people we can reach or how much it costs. For example, Unsellable Houses is cheaper per thousand target viewers, but Monday Night Football reaches a much larger audience. I would consider all three measures when choosing programs.

## Task 2.3 — Kestrel buyers

In [ ]:
# Calculate the index for Kestrel buyers.
program_summary["Kestrel index"] = (
    program_summary["kestrel_rating"]
    / program_summary["adult_rating"] * 100
)

# Compare both audience indexes.
index_table = program_summary[[
    "Program", "Index", "Kestrel index"
]].rename(columns={
    "Index": "Heavy-buyer index"
})

index_table = (
    index_table.set_index("Program")
    .loc[program_order]
    .reset_index()
)

display(index_table.style.format({
    "Heavy-buyer index": "{:.1f}",
    "Kestrel index": "{:.1f}"
}))

,Program,Heavy-buyer index,Kestrel index
0,Sunday Night Football (NBC),99.3,98.5
1,Monday Night Football (ABC),93.6,85.9
2,Dancing with the Stars (ABC),96.8,82.7
3,NCIS (CBS),110.1,56.5
4,Hannity (FNC),77.1,42.6
5,Unsellable Houses (HGTV),126.8,76.9
6,Halloween Wars (FOOD),116.2,95.2
7,Grotesquerie (FX),95.9,129.6


**Answer:**

The Kestrel index changes much more between programs because there are only 354 Kestrel buyers in the panel, compared to 1,661 heavy buyers. With such a small sample, even a few viewers can make a big difference in the ratings.

For example, NCIS has a heavy-buyer index of 110.1 but a Kestrel index of only 56.5. Grotesquerie shows the opposite pattern, with a Kestrel index of 129.6 compared to 95.9 for heavy buyers.

I would trust the heavy-buyer index more because it is based on a larger sample. To check the results, I would look at how many panel members actually watched each program, compare ratings across different weeks and, if possible, check them against another dataset.

## Task 3.1 — Weights

In [ ]:
# Convert smart-TV sessions to Eastern time.
acr = acr_sessions.copy()

acr["start_et"] = pd.to_datetime(
    acr["start_utc"], utc=True
).dt.tz_convert("America/New_York")

acr["end_et"] = pd.to_datetime(
    acr["end_utc"], utc=True
).dt.tz_convert("America/New_York")

acr["duration_min"] = (
    acr["end_et"] - acr["start_et"]
).dt.total_seconds() / 60

# Remove sessions shorter than one minute.
acr_clean = acr.loc[acr["duration_min"] >= 1].copy()

# Match each TV to exactly one household.
tv_ips = acr[["tv_id", "hashed_ip"]].drop_duplicates()

assert not tv_ips["tv_id"].duplicated().any()

households_per_ip = (
    hh_attributes.dropna(subset=["hashed_ip"])
    .groupby("hashed_ip")["household_id"]
    .nunique()
)

unique_ips = households_per_ip[
    households_per_ip == 1
].index

matched_tvs = (
    tv_ips.loc[tv_ips["hashed_ip"].isin(unique_ips)]
    .merge(
        hh_attributes.drop_duplicates("hashed_ip"),
        on="hashed_ip",
        how="inner",
        validate="many_to_one"
    )
)

# Prepare the 12 population cells.
universe_cells = universe.copy()

universe_cells["head_age"] = (
    universe_cells["head_age"].astype(str).str.strip()
)

universe_cells["income_band"] = (
    universe_cells["income_band"].astype(str).str.strip()
)

age_labels = universe_cells["head_age"].unique()

over_65 = next(
    x for x in age_labels
    if "65" in x and "under" not in x.lower()
)

under_65 = next(x for x in age_labels if x != over_65)

matched_tvs["head_age"] = np.where(
    matched_tvs["head_age_band"]
    .astype(str).str.contains("65")
    & ~matched_tvs["head_age_band"]
    .astype(str).str.lower().str.contains("under"),
    over_65,
    under_65
)

matched_tvs["income_band"] = (
    matched_tvs["income_band"].astype(str).str.strip()
)

matched_tvs = matched_tvs.merge(
    universe_cells,
    on=["head_age", "income_band"],
    how="left",
    validate="many_to_one"
)

assert matched_tvs["tv_households"].notna().all()

# Find matched TVs in-tab on each broadcast day.
acr_days = pd.date_range("2024-10-14", "2024-10-27")
tv_day_rows = []

for day in acr_days:
    day_start = (
        day + pd.Timedelta(hours=6)
    ).tz_localize("America/New_York")

    day_end = day_start + pd.Timedelta(days=1)

    day_sessions = acr_clean.loc[
        (acr_clean["start_et"] < day_end)
        & (acr_clean["end_et"] > day_start),
        ["tv_id"]
    ].drop_duplicates()

    day_sessions["broadcast_date"] = day
    tv_day_rows.append(day_sessions)

tv_days = pd.concat(tv_day_rows, ignore_index=True)

# Keep only TVs matched to one household.
in_tab = tv_days.merge(
    matched_tvs[
        ["tv_id", "head_age", "income_band", "tv_households"]
    ],
    on="tv_id",
    how="inner",
    validate="many_to_one"
)

# Calculate daily weights for each population cell.
cell_counts = (
    in_tab.groupby(
        ["broadcast_date", "head_age", "income_band"]
    )
    .size()
    .reset_index(name="matched_in_tab")
)

assert len(cell_counts) == len(acr_days) * len(universe_cells), (
    "Some population cells have no in-tab TVs."
)

cell_counts = cell_counts.merge(
    universe_cells,
    on=["head_age", "income_band"],
    how="left",
    validate="many_to_one"
)

cell_counts["tv_weight"] = (
    cell_counts["tv_households"]
    / cell_counts["matched_in_tab"]
)

in_tab = in_tab.drop(columns="tv_households").merge(
    cell_counts[
        ["broadcast_date", "head_age", "income_band", "tv_weight"]
    ],
    on=["broadcast_date", "head_age", "income_band"],
    how="left",
    validate="many_to_one"
)

# Check that daily weights add up to the TV universe.
daily_weight_sums = (
    in_tab.groupby("broadcast_date")["tv_weight"]
    .sum()
    .reindex(acr_days)
)

total_universe = universe_cells["tv_households"].sum()

assert np.allclose(daily_weight_sums, total_universe)

# Prepare the eight reference telecasts.
telecasts = reference_ratings.merge(
    programs,
    on="telecast_id",
    how="left",
    validate="one_to_one"
)

def to_eastern(value):
    timestamp = pd.Timestamp(value)
    if timestamp.tzinfo is None:
        return timestamp.tz_localize("America/New_York")
    return timestamp.tz_convert("America/New_York")

# Only edited live sessions count as viewing.
acr_live = acr_clean.loc[
    acr_clean["session_type"].astype(str).str.lower().eq("live")
].copy()

# Calculate unweighted and weighted ratings.
rating_rows = []

for telecast in telecasts.itertuples(index=False):
    start = to_eastern(telecast.start_ts)
    end = to_eastern(telecast.end_ts)

    day = pd.Timestamp(telecast.broadcast_date).normalize()
    if day.tzinfo is not None:
        day = day.tz_localize(None)

    day_tvs = in_tab.loc[
        in_tab["broadcast_date"] == day,
        ["tv_id", "tv_weight"]
    ]

    sessions = acr_live.loc[
        (acr_live["network_code"] == telecast.network_code)
        & (acr_live["start_et"] < end)
        & (acr_live["end_et"] > start)
    ].copy()

    sessions["overlap_min"] = (
        sessions["end_et"].clip(upper=end)
        - sessions["start_et"].clip(lower=start)
    ).dt.total_seconds() / 60

    minutes_per_tv = (
        sessions.groupby("tv_id")["overlap_min"].sum()
    )

    watched_ids = set(
        minutes_per_tv[minutes_per_tv >= 1].index
    )

    day_tvs = day_tvs.copy()
    day_tvs["watched"] = day_tvs["tv_id"].isin(watched_ids)

    unweighted_rating = (
        day_tvs["watched"].mean() * 100
    )

    weighted_rating = (
        day_tvs.loc[day_tvs["watched"], "tv_weight"].sum()
        / day_tvs["tv_weight"].sum() * 100
    )

    rating_rows.append({
        "Telecast": (
            f"{telecast.program_name} ({telecast.network_code})"
        ),
        "Unweighted (%)": unweighted_rating,
        "Weighted (%)": weighted_rating,
        "Reference (%)": telecast.hh_rating
    })

weighted_ratings = pd.DataFrame(rating_rows)

# Calculate the mean absolute gaps.
mean_gap_unweighted = (
    weighted_ratings["Unweighted (%)"]
    - weighted_ratings["Reference (%)"]
).abs().mean()

mean_gap_weighted = (
    weighted_ratings["Weighted (%)"]
    - weighted_ratings["Reference (%)"]
).abs().mean()

share_closed = (
    (mean_gap_unweighted - mean_gap_weighted)
    / mean_gap_unweighted
)

# Required results.
print("TVs matched to one household:", len(matched_tvs))
print("Weights sum per day (M):", f"{daily_weight_sums.mean() / 1e6:.1f}")

display(weighted_ratings.style.format({
    "Unweighted (%)": "{:.2f}",
    "Weighted (%)": "{:.2f}",
    "Reference (%)": "{:.2f}"
}))

print("Mean gap unweighted (points):", f"{mean_gap_unweighted:.2f}")
print("Mean gap weighted (points):", f"{mean_gap_weighted:.2f}")
print("Share of gap closed:", f"{share_closed:.3f}")

TVs matched to one household: 19607
Weights sum per day (M): 125.8


,Telecast,Unweighted (%),Weighted (%),Reference (%)
0,MLB Postseason (FOX),17.03,17.76,25.86
1,NFL Football (NBC),22.62,23.75,33.14
2,NCIS (CBS),5.50,5.80,10.56
3,Hannity (FNC),1.04,1.35,2.30
4,Unsellable Houses (HGTV),2.12,2.08,3.69
5,Halloween Baking Championship (FOOD),0.95,0.90,1.63
6,Grotesquerie (FX),2.17,2.11,3.53
7,The Real Housewives of New York City (BRAVO),0.71,0.64,1.23


Mean gap unweighted (points): 3.73
Mean gap weighted (points): 3.44
Share of gap closed: 0.075


## Task 3.2 — What weighting cannot fix

**Answer:**

After weighting, the ratings increased for NFL Football, MLB Postseason, NCIS and Hannity, while most of the smaller cable programs went down slightly. This happened because weighting changed how much influence different household groups had on the results.

However, the improvement was quite small. The average gap decreased from 3.73 to 3.44 points, so weighting only closed about 7.5% of it. It cannot recover viewing that the smart TVs failed to record or fix differences between people who participate in the smart-TV sample and the general population.

Weighting by income barely changes the results because income differences do not explain enough of the viewing differences in this sample. So even after weighting, the smart-TV ratings are still noticeably lower than the reference ratings.

## Task 4.1 — October predicts October

In [ ]:
# Select prime-time programs for the forecast.
forecast_networks = ["CBS", "NBC", "ABC", "FOX", "FNC"]

forecast_prime = prime.loc[
    prime["network_code"].isin(forecast_networks)
    & prime["broadcast_date"].between(
        "2024-10-01", "2024-10-28"
    )
].copy()

# Split the data into history and forecast week.
history = forecast_prime.loc[
    forecast_prime["broadcast_date"] <= "2024-10-21"
].copy()

forecast_week = forecast_prime.loc[
    forecast_prime["broadcast_date"] >= "2024-10-22"
].copy()

# Keep only programs that aired during the history period.
history_keys = set(zip(
    history["network_code"], history["program_name"]
))

forecast_week = forecast_week.loc[
    forecast_week["program_key"].isin(history_keys)
].copy()

forecast_keys = set(forecast_week["program_key"])

history = history.loc[
    history["program_key"].isin(forecast_keys)
].copy()

telecasts_to_rate = pd.concat(
    [history, forecast_week],
    ignore_index=True
).drop_duplicates("telecast_id")

# Prepare daily weights for persons aged 2+.
p2_ids = set(
    ages.loc[ages["age"] >= 2, "RESPID"]
)

heavy_ids = set(
    purchase_panel.loc[
        purchase_panel["pairs"] == "3+", "RESPID"
    ]
)

panel_weights = daily_weights.loc[
    daily_weights["resp_type"].isin(["R", "L"]),
    ["respondent_id", "broadcast_date", "weight"]
].copy()

panel_weights["RESPID"] = normalize_ids(
    panel_weights["respondent_id"]
)

panel_weights["broadcast_date"] = pd.to_datetime(
    panel_weights["broadcast_date"]
).dt.normalize()

panel_weights = panel_weights.loc[
    panel_weights["RESPID"].isin(p2_ids)
    & panel_weights["broadcast_date"].between(
        "2024-10-01", "2024-10-28"
    )
].copy()

panel_weights["heavy"] = (
    panel_weights["RESPID"].isin(heavy_ids)
)

weights_by_day = {
    day: data.set_index("RESPID")
    for day, data in panel_weights.groupby("broadcast_date")
}

# Prepare live viewing sessions for the five networks.
forecast_viewings = viewings.loc[
    viewings["resp_type"].isin(["R", "L"])
    & viewings["network_code"].isin(forecast_networks)
    & viewings["playback"].astype(str).str.lower().eq("live"),
    ["respondent_id", "network_code", "start_ts", "end_ts"]
].copy()

forecast_viewings["RESPID"] = normalize_ids(
    forecast_viewings["respondent_id"]
)

for col in ["start_ts", "end_ts"]:
    forecast_viewings[col] = eastern_time(
        forecast_viewings[col]
    )

# Group viewing sessions by broadcast day and network.
forecast_viewings["view_date"] = (
    forecast_viewings["start_ts"] - pd.Timedelta(hours=6)
).dt.normalize()

viewings_by_day = {
    key: data[["RESPID", "start_ts", "end_ts"]]
    for key, data in forecast_viewings.groupby(
        ["network_code", "view_date"]
    )
}

# Calculate actual ratings for each telecast.
rating_rows = []

for telecast in telecasts_to_rate.itertuples(index=False):
    day = telecast.broadcast_date
    start = telecast.start_ts
    end = telecast.end_ts

    duration = (end - start).total_seconds() / 60

    # Include sessions that may overlap across broadcast days.
    parts = [
        viewings_by_day.get((
            telecast.network_code,
            day + pd.Timedelta(days=offset)
        ))
        for offset in [-1, 0, 1]
    ]

    parts = [part for part in parts if part is not None]

    if parts:
        sessions = pd.concat(parts, ignore_index=True)

        sessions = sessions.loc[
            (sessions["start_ts"] < end)
            & (sessions["end_ts"] > start)
        ].copy()

        sessions["minutes"] = (
            sessions["end_ts"].clip(upper=end)
            - sessions["start_ts"].clip(lower=start)
        ).dt.total_seconds() / 60

        minutes_per_person = (
            sessions.groupby("RESPID")["minutes"]
            .sum()
            .clip(upper=duration)
        )
    else:
        minutes_per_person = pd.Series(dtype=float)

    panel_day = weights_by_day[day]

    viewers = panel_day.join(
        minutes_per_person.rename("minutes"),
        how="inner"
    )

    # Persons 2+ rating.
    p2_viewers = (
        viewers["weight"] * viewers["minutes"]
    ).sum() / duration

    p2_universe = panel_day["weight"].sum()

    p2_rating = p2_viewers / p2_universe * 100

    # Heavy-buyer rating.
    heavy_viewers = viewers.loc[viewers["heavy"]]

    heavy_avg_viewers = (
        heavy_viewers["weight"] * heavy_viewers["minutes"]
    ).sum() / duration

    heavy_universe = panel_day.loc[
        panel_day["heavy"], "weight"
    ].sum()

    heavy_rating = heavy_avg_viewers / heavy_universe * 100

    rating_rows.append({
        "telecast_id": telecast.telecast_id,
        "p2_rating": p2_rating,
        "heavy_rating": heavy_rating
    })

actual_ratings = pd.DataFrame(rating_rows)

rated_telecasts = telecasts_to_rate.merge(
    actual_ratings,
    on="telecast_id",
    validate="one_to_one"
)

# Calculate average history ratings for each program.
history_ratings = rated_telecasts.loc[
    rated_telecasts["broadcast_date"] <= "2024-10-21"
]

history_means = (
    history_ratings.groupby(
        ["network_code", "program_name"]
    )[["p2_rating", "heavy_rating"]]
    .mean()
    .rename(columns={
        "p2_rating": "p2_forecast",
        "heavy_rating": "heavy_forecast"
    })
)

# Attach the forecasts to October 22-28 telecasts.
forecast_table = rated_telecasts.loc[
    rated_telecasts["broadcast_date"] >= "2024-10-22"
].copy()

forecast_table = forecast_table.join(
    history_means,
    on=["network_code", "program_name"]
)

# Calculate absolute percentage errors.
forecast_table["p2_error"] = (
    (forecast_table["p2_rating"] - forecast_table["p2_forecast"])
    .abs()
    / forecast_table["p2_rating"].replace(0, np.nan)
    * 100
)

forecast_table["heavy_error"] = (
    (forecast_table["heavy_rating"] - forecast_table["heavy_forecast"])
    .abs()
    / forecast_table["heavy_rating"].replace(0, np.nan)
    * 100
)

mape_p2 = forecast_table["p2_error"].mean()
mape_heavy = forecast_table["heavy_error"].mean()

# Find the largest P2+ forecast error.
biggest_miss = forecast_table.loc[
    forecast_table["p2_error"].idxmax()
]

biggest_miss_name = (
    f"{biggest_miss['program_name']} "
    f"({biggest_miss['network_code']})"
)

biggest_miss_date = (
    biggest_miss["broadcast_date"].strftime("%Y-%m-%d")
)

# Required results.
print("Telecasts forecast (n):", len(forecast_table))
print("MAPE, P2+ (%):", f"{mape_p2:.1f}")
print("MAPE, heavy buyers (%):", f"{mape_heavy:.1f}")
print(
    "Biggest miss, P2+ (program, date):",
    biggest_miss_name, biggest_miss_date
)

Telecasts forecast (n): 101
MAPE, P2+ (%): 35.8
MAPE, heavy buyers (%): 38.6
Biggest miss, P2+ (program, date): Krapopolis (FOX) 2024-10-27


## Task 4.2 — Why it missed

In [ ]:
# Find the five largest P2+ forecast errors.
biggest_misses = (
    forecast_table.dropna(subset=["p2_error"])
    .nlargest(5, "p2_error")
    .copy()
)

biggest_misses["Program"] = (
    biggest_misses["program_name"]
    + " (" + biggest_misses["network_code"] + ")"
)

biggest_misses["Date"] = (
    biggest_misses["broadcast_date"].dt.strftime("%Y-%m-%d")
)

# Display the five biggest misses.
misses_table = biggest_misses[[
    "Program", "Date", "p2_forecast",
    "p2_rating", "p2_error"
]].rename(columns={
    "p2_forecast": "Forecast (%)",
    "p2_rating": "Actual (%)",
    "p2_error": "Error (%)"
})

display(misses_table.style.format({
    "Forecast (%)": "{:.2f}",
    "Actual (%)": "{:.2f}",
    "Error (%)": "{:.1f}"
}))

# Check the TV schedule around the missed programs.
schedule = programs.copy()

schedule["broadcast_date"] = pd.to_datetime(
    schedule["broadcast_date"]
).dt.normalize()

for col in ["start_ts", "end_ts"]:
    schedule[col] = eastern_time(schedule[col])

schedule_rows = []

for telecast in biggest_misses.itertuples(index=False):
    same_day = schedule.loc[
        (schedule["broadcast_date"] == telecast.broadcast_date)
        & (schedule["network_code"] == telecast.network_code)
    ].sort_values("start_ts")

    previous = same_day.loc[
        same_day["end_ts"] <= telecast.start_ts
    ].tail(3)

    next_program = same_day.loc[
        same_day["start_ts"] >= telecast.end_ts
    ].head(1)

    schedule_rows.append({
        "Program": f"{telecast.program_name} ({telecast.network_code})",
        "Date": telecast.broadcast_date.strftime("%Y-%m-%d"),
        "Earlier programs": ", ".join(previous["program_name"]),
        "Next program": (
            next_program.iloc[0]["program_name"]
            if not next_program.empty else "None"
        )
    })

display(pd.DataFrame(schedule_rows))

,Program,Date,Forecast (%),Actual (%),Error (%)
335,Krapopolis (FOX),2024-10-27,1.06,0.22,387.8
333,Universal Basic Guys (FOX),2024-10-27,0.68,0.26,162.2
292,Fire Country (CBS),2024-10-27,1.70,0.67,154.8
320,Accused (FOX),2024-10-22,0.95,0.37,154.6
259,What Would You Do? (ABC),2024-10-23,0.44,0.18,138.1


,Program,Date,Earlier programs,Next program
0,Krapopolis (FOX),2024-10-27,"The Simpsons, Universal Basic Guys, Bob's Burgers",Bob's Burgers
1,Universal Basic Guys (FOX),2024-10-27,"NFL Football, Local / Syndicated Programming, ...",Bob's Burgers
2,Fire Country (CBS),2024-10-27,"Fire Country, Tracker, The Equalizer",Local / Syndicated Programming
3,Accused (FOX),2024-10-22,Murder in a Small Town,Universal Basic Guys
4,What Would You Do? (ABC),2024-10-23,"Local / Syndicated Programming, The Golden Bac...",Local / Syndicated Programming


**Answer:**

The biggest forecast error was for Krapopolis (FOX) on October 27. The predicted rating was 1.06%, but the actual rating was only 0.22%, giving an error of 387.8%.

Two of the biggest misses happened on FOX that day. NFL Football and local programming may have affected the shows that followed. Fire Country, Accused and What Would You Do? also performed worse than expected, possibly because of changes in the schedule or the programs airing before them.

The forecast was less accurate for heavy buyers (38.6% MAPE compared to 35.8% for P2+) because there are fewer people in this audience, making the ratings less stable.

To improve the forecast, I would consider sports events and the programs airing before each show. I would also use more historical ratings instead of just three weeks.

## Task 5.1 — The plan

In [ ]:
# Allocate the advertising budget.
budget = 1_000_000

spot_plan = pd.DataFrame({
    "Program": [
        "Sunday Night Football (NBC)",
        "Dancing with the Stars (ABC)",
        "NCIS (CBS)",
        "Hannity (FNC)",
        "Unsellable Houses (HGTV)",
        "Grotesquerie (FX)"
    ],
    "Spots": [1, 1, 1, 3, 3, 2]
})

# Add costs and target viewers from Task 2.1.
spot_plan = spot_plan.merge(
    program_summary[["Program", "cost_30", "heavy_viewers"]],
    on="Program",
    how="left",
    validate="one_to_one"
)

assert spot_plan["cost_30"].notna().all()

# Calculate cost and target impressions.
spot_plan["Cost ($)"] = (
    spot_plan["Spots"] * spot_plan["cost_30"]
)

spot_plan["Target impressions (M)"] = (
    spot_plan["Spots"] * spot_plan["heavy_viewers"] / 1_000_000
)

plan_cost = spot_plan["Cost ($)"].sum()
plan_impressions = spot_plan["Target impressions (M)"].sum()

assert plan_cost <= budget, "The plan exceeds the budget."

plan_table = spot_plan[[
    "Program", "Spots", "Cost ($)", "Target impressions (M)"
]]

display(plan_table.style.format({
    "Cost ($)": "{:,.0f}",
    "Target impressions (M)": "{:.3f}"
}))

print("Plan cost ($):", f"{plan_cost:,.0f}")
print("Plan target impressions (M):", f"{plan_impressions:.1f}")

,Program,Spots,Cost ($),Target impressions (M)
0,Sunday Night Football (NBC),1,"487,200",2.865
1,Dancing with the Stars (ABC),1,"158,600",1.564
2,NCIS (CBS),1,"178,800",1.206
3,Hannity (FNC),3,"119,700",1.251
4,Unsellable Houses (HGTV),3,"38,700",0.952
5,Grotesquerie (FX),2,"15,184",0.248


Plan cost ($): 998,184
Plan target impressions (M): 8.1


## Task 5.2 — Recommendation

**Answer:**

I would spend USD 998,184 on 11 spots across six programs, expecting about 8.1 million impressions among heavy athletic-shoe buyers. Sunday Night Football delivers a large audience with one spot, while Dancing with the Stars and NCIS provide additional impressions across different programs. I also included Hannity, Unsellable Houses and Grotesquerie because they have lower target CPMs, especially Unsellable Houses at USD 41. Unsellable Houses also has the highest heavy-buyer index (126.8), but I would not choose programs based on index alone.

The target composition ranges from 15.2% to 24.9%, so most impressions would still reach people outside our target audience. Based on these shares, approximately USD 804,000 of the budget would correspond to non-target impressions. I would be careful with the estimates because the heavy-buyer forecast had a MAPE of 38.6%, and I would try to negotiate an audience guarantee with make-goods if the campaign under-delivers. After the campaign, I would compare the actual target impressions, CPM and audience composition with the planned results.